In [0]:
#imports
from pyspark.sql.functions import current_timestamp,col

#project configs
BASE_DIR="/Volumes/aarohan_bank/bronze/aarohan_landing"
SCHEMA_BASE=f"{BASE_DIR}/_schema"
CHECKPOINT_BASE=f"{BASE_DIR}/_checkpoints"
TARGET_CATALOG="aarohan_bank"
TARGET_SCHEMA="bronze"

In [0]:
#Discover source files
files=dbutils.fs.ls(BASE_DIR)

csv_file_paths_list=[]

for file in files:
    if file.name.lower().endswith(".csv"):
        csv_file_paths_list.append(file.path)
csv_file_paths_list.sort()

print(f"CSV files discovered: {len(csv_file_paths_list)}")

In [0]:
def ingest_table(file_path):
    #config
    FILE_NAME=file_path.split("/")[-1]
    TABLE_NAME=FILE_NAME.split(".")[0]
    SCHEMA_PATH=f"{SCHEMA_BASE}/{TABLE_NAME}"
    CHECKPOINT_PATH=f"{CHECKPOINT_BASE}/{TABLE_NAME}"
    TARGET_TABLE=f"{TARGET_CATALOG}.{TARGET_SCHEMA}.{TABLE_NAME}"
    print("=" * 60)
    print(f"Starting Ingestion:{TABLE_NAME}")

    #read the files
    df=(
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format","csv")
        .option("inferColumnTypes","true")
        .option("cloudFiles.schemaLocation",SCHEMA_PATH)
        .option("pathGlobFilter",f"{TABLE_NAME}.csv")
        .option("header","true")
        .load(BASE_DIR)
    )

    #adding ingestion metadata
    df=(
        df
        .withColumn("_ingested_at",current_timestamp())
        .withColumn("_source_file",col("_metadata.file_name"))
    )
    
    #write to bronze delta table
    query=(df.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation",CHECKPOINT_PATH)
        .trigger(availableNow=True)
        .toTable(TARGET_TABLE)
    )

    #wait for ingestion to finish
    query.awaitTermination()

    print(f"Completed Ingestion: {TABLE_NAME}")


In [0]:
successful_tables=[]
failed_tables=[]

for file_path in csv_file_paths_list:
    table_name=file_path.split("/")[-1].split(".")[0]
    try:
        ingest_table(file_path)
        successful_tables.append(table_name)
    except Exception as e:
        failed_tables.append(table_name)
        print("="*60)
        print(f"Failed:{table_name}")
        print(f"ERROR: {e}")
        print("="*60)

print("="*60)
print("BRONZE INGESTION COMPLETED")
print("="*60)

print(f"TOTAL FILES: {len(csv_file_paths_list)}")
print(f"SUCCESSFUL: {len(successful_tables)}")
print(f"FAILED: {len(failed_tables)}")
            